# Lesson 16: Streaming Arithmetic Answers

## Objective
Use `graph.stream()` to receive **real-time updates** as each node executes, and stream **LLM token-by-token output** to the user.

## Limitation of Lesson 15
- ❌ `invoke()` blocked until the entire graph finished
- ❌ No visibility into which node was running
- ❌ LLM responses appeared all at once (no streaming tokens)

## What is NEW in Lesson 16?
- ✅ `graph.stream()` — iterate over state updates as each node completes
- ✅ Stream modes: `"updates"`, `"values"`, `"debug"`
- ✅ `graph.astream_events()` — async streaming with fine-grained events
- ✅ LLM token-by-token streaming with `.stream()` on the LLM
- ✅ Understanding different stream modes

## Theory: Stream Modes

| Mode | What you receive |
|------|----------------|
| `"updates"` | Dict of `{node_name: state_update}` after each node |
| `"values"` | Full state snapshot after each node |
| `"debug"` | Detailed execution events (tasks, results, metadata) |

```python
for event in graph.stream(state, stream_mode="updates"):
    node_name = list(event.keys())[0]
    state_update = event[node_name]
    print(f"Node {node_name} updated: {state_update}")
```


In [ ]:
# ── Cell 1: Imports + LLM Setup ──────────────────────────────────────────────
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

import os, time
import vertexai
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

from langgraph.graph import StateGraph, MessagesState, START, END
from typing import TypedDict, Optional, Literal, Annotated
import operator
from IPython.display import Image, display, clear_output

load_dotenv()
vertexai.init(project=os.getenv("PROJECT_ID"), location=os.getenv("LOCATION"))

# Regular LLM for standard use
llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0)

# Streaming LLM — same model, streaming=True enables token-by-token output
llm_streaming = ChatVertexAI(model="gemini-2.5-flash", temperature=0, streaming=True)

print("✓ Setup complete")
print("  llm:          standard (full response at once)")
print("  llm_streaming: streaming (token by token)")


In [ ]:
# ── Cell 2: State Schema ────────────────────────────────────────────────────
class State(TypedDict):
    question: str
    a: Optional[float]
    b: Optional[float]
    operation: Optional[str]
    result: Optional[float]
    explanation: str
    history: Annotated[list, operator.add]

print("✓ State defined")


In [ ]:
# ── Cell 3: Graph Nodes ──────────────────────────────────────────────────────
PARSE_PROMPT = """Parse this arithmetic question. Respond EXACTLY:
OPERATION: <add|multiply|divide>
A: <number>
B: <number>"""

def parse_node(state: State) -> dict:
    print(f"  [parse] Parsing: '{state['question']}'")
    resp = llm.invoke([
        SystemMessage(content=PARSE_PROMPT),
        HumanMessage(content=state["question"])
    ])
    lines = {l.split(":")[0].strip(): l.split(":",1)[1].strip()
             for l in resp.content.strip().split("\n") if ":" in l}
    op = lines.get("OPERATION","add").lower()
    a  = float(lines.get("A","0"))
    b  = float(lines.get("B","0"))
    print(f"  [parse] Extracted: {a} {op} {b}")
    return {"operation": op, "a": a, "b": b, "history": [f"parsed: {a} {op} {b}"]}

def compute_node(state: State) -> dict:
    a, b, op = state["a"], state["b"], state["operation"]
    if op == "add":      res, sym = a+b, "+"
    elif op == "multiply": res, sym = a*b, "×"
    else:                res, sym = (a/b if b!=0 else None), "÷"
    entry = f"{a} {sym} {b} = {res}"
    print(f"  [compute] {entry}")
    return {"result": res, "history": [entry]}

def explain_node(state: State) -> dict:
    # This node uses STREAMING LLM — generates explanation token by token
    a, b, op, res = state["a"], state["b"], state["operation"], state["result"]
    prompt = f"Explain in 2 sentences how to compute {a} {op} {b} = {res}. Be educational."
    
    print("  [explain] Streaming explanation: ", end="", flush=True)
    
    # Stream tokens from LLM
    full_text = ""
    for chunk in llm_streaming.stream([HumanMessage(content=prompt)]):
        token = chunk.content
        print(token, end="", flush=True)
        full_text += token
    print()  # newline after streaming
    
    return {"explanation": full_text, "history": [f"explained: {full_text[:40]}..."]}

print("✓ Three nodes defined (explain_node uses streaming LLM)")


In [ ]:
# ── Cell 4: Build Graph ──────────────────────────────────────────────────────
builder = StateGraph(State)
builder.add_node("parse",   parse_node)
builder.add_node("compute", compute_node)
builder.add_node("explain", explain_node)

builder.add_edge(START,     "parse")
builder.add_edge("parse",   "compute")
builder.add_edge("compute", "explain")
builder.add_edge("explain", END)

graph = builder.compile()
print("✓ Graph compiled")


In [ ]:
# ── Cell 5: Visualize ───────────────────────────────────────────────────────
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
# ── Cell 6: Stream Mode = "updates" ─────────────────────────────────────────
# Receive {node_name: partial_state_update} after each node completes

print("="*55)
print("STREAM MODE: 'updates'")
print("Receive: {node_name: state_update} after each node")
print("="*55)

state = {"question": "what is 9 multiplied by 7?",
         "a": None, "b": None, "operation": None,
         "result": None, "explanation": "", "history": []}

for event in graph.stream(state, stream_mode="updates"):
    node_name = list(event.keys())[0]
    update    = event[node_name]
    print(f"\n── Node '{node_name}' completed ──────────────────")
    for k, v in update.items():
        val_str = str(v)[:60] + ("..." if len(str(v)) > 60 else "")
        print(f"   {k}: {val_str}")


In [ ]:
# ── Cell 7: Stream Mode = "values" ──────────────────────────────────────────
# Receive FULL state snapshot after each node completes

print("="*55)
print("STREAM MODE: 'values'")
print("Receive: full state snapshot after each node")
print("="*55)

state = {"question": "divide 144 by 12",
         "a": None, "b": None, "operation": None,
         "result": None, "explanation": "", "history": []}

print("Watching state evolution:")
for i, full_state in enumerate(graph.stream(state, stream_mode="values")):
    print(f"\n── After step {i} ─────────────────────────────────")
    print(f"   operation = {full_state.get('operation')}")
    print(f"   a={full_state.get('a')}, b={full_state.get('b')}")
    print(f"   result    = {full_state.get('result')}")
    print(f"   history   = {full_state.get('history')}")


In [ ]:
# ── Cell 8: LLM Token Streaming Demo ─────────────────────────────────────────
# Show token-by-token streaming from the LLM directly
print("="*55)
print("LLM TOKEN STREAMING DEMO")
print("="*55)
print("Streaming explanation token by token:\n")

prompt = "Explain step by step how to multiply 13 by 7. Be detailed."
print("Answer: ", end="", flush=True)
for chunk in llm_streaming.stream([HumanMessage(content=prompt)]):
    print(chunk.content, end="", flush=True)
print("\n\n✓ Streaming complete")


In [ ]:
# ── Cell 9: Stream with Node Timing ──────────────────────────────────────────
# Track how long each node takes

import time

print("="*55)
print("STREAMING WITH TIMING")
print("="*55)

state = {"question": "what is 15 plus 28?",
         "a": None, "b": None, "operation": None,
         "result": None, "explanation": "", "history": []}

start_total = time.time()
for event in graph.stream(state, stream_mode="updates"):
    node_name = list(event.keys())[0]
    elapsed = time.time() - start_total
    print(f"  [{elapsed:.2f}s] Node '{node_name}' completed")

total = time.time() - start_total
print(f"\n  Total execution time: {total:.2f}s")


## Stream Modes Summary

```python
# Mode 1: updates — most common for production
for event in graph.stream(state, stream_mode="updates"):
    node = list(event.keys())[0]
    update = event[node]

# Mode 2: values — full state at each step
for full_state in graph.stream(state, stream_mode="values"):
    result = full_state.get("result")

# Mode 3: debug — every internal event
for event in graph.stream(state, stream_mode="debug"):
    print(event["type"], event.get("step"))
```

### LLM Token Streaming
```python
# ChatVertexAI with streaming=True
llm_s = ChatVertexAI(model="gemini-2.5-flash", streaming=True)
for chunk in llm_s.stream([HumanMessage(content="...")]):
    print(chunk.content, end="", flush=True)
```

## Summary

| Feature | `invoke()` | `stream()` |
|---------|-----------|-----------|
| Output timing | All at end | After each node |
| Visibility | None | Real-time |
| LLM tokens | Whole response | Token by token |

## Limitations → What Lesson 17 Solves
- ❌ If the graph crashes mid-run, all progress is lost
- ❌ Long-running agents need to be restartable from any point
- ❌ **Lesson 17**: Checkpointing — save state after every node, resume from any step
